# 5.12 Derinlemesine: Gauss Karışımları

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/05-sklearn/12-gaussian-mixtures.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 05.12 Gaussian Mixtures

Önceki bölümde incelediğimiz k-means kümeleme modeli basit ve anlaşılması nispeten kolaydır; ancak bu basitlik uygulamada pratik zorluklara yol açar. Özellikle k-means'in olasılıksal olmaması ve küme merkezine basit uzaklıkla üyelik ataması, birçok gerçek dünya durumunda zayıf performansa neden olur.

Bu bölümde Gauss karışım modellerine bakacağız; bunlar k-means fikirlerinin bir uzantısı olarak görülebilir, aynı zamanda basit kümelemenin ötesinde güçlü bir tahmin aracı da olabilir.

Standart içe aktarmalarla başlayalım:


In [ ]:
# imports_gmm.py
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
import numpy as np



## Gauss Karışımlarını Motive Etmek: k-Means'in Zayıflıkları


In [ ]:
# Generate some data
from sklearn.datasets import make_blobs
X, y_true = make_blobs(n_samples=400, centers=4,
                       cluster_std=0.60, random_state=0)
X = X[:, ::-1] # flip axes for better plotting



In [ ]:
# Plot the data with k-means labels
from sklearn.cluster import KMeans
kmeans = KMeans(4, random_state=0)
labels = kmeans.fit(X).predict(X)
plt.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis');



Sezgisel olarak bazı noktalar için küme atamasının diğerlerinden daha kesin olmasını bekleriz: ortadaki iki küme arasında hafif örtüşme var gibi görünüyor; aralarındaki noktaların küme atamasına tam güvenmeyebiliriz. Ne yazık ki k-means modelinin küme atamalarının olasılık veya belirsizliğini ölçen içsel bir ölçüsü yoktur (bootstrap ile tahmin edilebilir). Bunun için modeli genelleştirmeyi düşünmeliyiz.

k-means modelini, her kümenin merkezine bir daire (yüksek boyutlarda hiperküre) yerleştiren ve yarıçapını kümedeki en uzak noktayla tanımlayan bir model olarak düşünebiliriz. Bu yarıçap eğitim kümesinde sert bir kesim görevi görür: daire dışındaki noktalar küme üyesi sayılmaz. Bu küme modelini aşağıdaki fonksiyonla görselleştirebiliriz (aşağıdaki şekil):


In [ ]:
# plot_kmeans_fn.py
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

def plot_kmeans(kmeans, X, n_clusters=4, rseed=0, ax=None):
    labels = kmeans.fit_predict(X)

    # plot the input data
    ax = ax or plt.gca()
    ax.axis('equal')
    ax.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis', zorder=2)

    # plot the representation of the KMeans model
    centers = kmeans.cluster_centers_
    radii = [cdist(X[labels == i], [center]).max()
             for i, center in enumerate(centers)]
    for c, r in zip(centers, radii):
        ax.add_patch(plt.Circle(c, r, ec='black', fc='lightgray',
                                lw=3, alpha=0.5, zorder=1))



In [ ]:
# kmeans_circles.py
kmeans = KMeans(n_clusters=4, random_state=0)
plot_kmeans(kmeans, X)



k-means için önemli bir gözlem: bu küme modelleri dairesel olmak zorundadır — k-means elips veya uzun kümeleri hesaba katamaz. Aynı veriyi dönüştürürsek küme atamaları karışır (aşağıdaki şekil):


In [ ]:
# kmeans_stretched.py
rng = np.random.RandomState(13)
X_stretched = np.dot(X, rng.randn(2, 2))

kmeans = KMeans(n_clusters=4, random_state=0)
plot_kmeans(kmeans, X_stretched)



Gözle bu dönüştürülmüş kümelerin dairesel olmadığını görürüz; dairesel kümeler kötü bir uyum olur. Yine de k-means buna uyum sağlayamaz ve veriyi dört dairesel kümeye zorlar; sonuçta daireler örtüşür — özellikle grafiğin sağ alt köşesine bakın.

Bu özel durumu PCA ile ön işlemeyle ele almayı düşünebiliriz (bkz. 5.9 PCA); ancak pratikte böyle küresel bir işlemin bireysel grupları daireselleştireceğinin garantisi yoktur.

k-means'in bu iki dezavantajı — küme şeklinde esneklik eksikliği ve olasılıksal atama eksikliği — birçok veri kümesinde (özellikle düşük boyutlu) umut ettiğiniz kadar iyi performans vermeyebileceği anlamına gelir.

Belirsizliği, her noktanın yalnızca en yakın merkeze değil tüm küme merkezlerine uzaklıklarını karşılaştırarak ölçerek; küme sınırlarının daire yerine elips olmasına izin vererek genelleştirebilirsiniz. Bunlar Gauss karışım modellerinin iki temel bileşenidir.

## E–M'yi Genelleştirmek: Gauss Karışım Modelleri

Gauss karışım modeli (GMM), girdi veri kümesini en iyi modelleyen çok boyutlu Gauss olasılık dağılımları karışımını bulmaya çalışır. En basit durumda GMM, k-means ile aynı şekilde küme bulmak için kullanılabilir (aşağıdaki şekil):


In [ ]:
# gmm_fit_predict.py
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=4).fit(X)
labels = gmm.predict(X)
plt.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis');



Ancak GMM kaputun altında olasılıksal bir model içerdiğinden, olasılıksal küme atamaları da bulunabilir — Scikit-Learn'de predict_proba yöntemiyle. Bu, herhangi bir noktanın verilen kümeye ait olasılığını ölçen [n_samples, n_clusters] boyutunda bir matris döndürür:


In [ ]:
# gmm_predict_proba.py
probs = gmm.predict_proba(X)
print(probs[:5].round(3))



### 🧪 Şimdi deneyin

🧪 
      GMM ile olasılıksal küme atamasını inceleyin:
          
      import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
X, _ = make_blobs(n_samples=200, centers=3, random_state=0)
gmm = GaussianMixture(n_components=3, random_state=0).fit(X)
print(gmm.predict_proba(X[:3]).round(3))

Bu belirsizliği, örneğin her noktanın boyutunu tahmin kesinliğiyle orantılı yaparak görselleştirebiliriz; aşağıdaki şekilde kümeler arasındaki sınırdaki noktaların tam da bu atama belirsizliğini yansıttığını görürüz:


In [ ]:
# gmm_uncertainty_size.py
size = 50 * probs.max(1) ** 2  # square emphasizes differences
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=size);



Kaputun altında Gauss karışım modeli k-means'e çok benzer: beklenti–maksimizasyon kullanır; nitel olarak şunları yapar:

Sonuçta her küme sert kenarlı bir küreyle değil, düzgün bir Gauss modeliyle ilişkilidir. k-means E–M'de olduğu gibi algoritma bazen küresel optimumu kaçırabilir; pratikte birden fazla rastgele başlangıç kullanılır.

GMM çıktısına dayanarak elipsler çizen ve küme konumlarını/şekillerini görselleştirmemize yardımcı olacak bir fonksiyon yazalım:


In [ ]:
# draw_ellipse_fn.py
from matplotlib.patches import Ellipse

def draw_ellipse(position, covariance, ax=None, **kwargs):
    """Draw an ellipse with a given position and covariance"""
    ax = ax or plt.gca()
    
    # Convert covariance to principal axes
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)
    
    # Draw the ellipse
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle, **kwargs))
        
def plot_gmm(gmm, X, label=True, ax=None):
    ax = ax or plt.gca()
    labels = gmm.fit(X).predict(X)
    if label:
        ax.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis', zorder=2)
    else:
        ax.scatter(X[:, 0], X[:, 1], s=40, zorder=2)
    ax.axis('equal')
    
    w_factor = 0.2 / gmm.weights_.max()
    for pos, covar, w in zip(gmm.means_, gmm.covariances_, gmm.weights_):
        draw_ellipse(pos, covar, alpha=w * w_factor)



Bununla birlikte dört bileşenli GMM'nin başlangıç verimize ne verdiğine bakalım (aşağıdaki şekil):


In [ ]:
# plot_gmm_four.py
gmm = GaussianMixture(n_components=4, random_state=42)
plot_gmm(gmm, X)



Benzer şekilde GMM ile gerilmiş veri kümemize uyum sağlayabiliriz; tam kovaryansa izin verildiğinde model çok uzun, gerilmiş kümeleri bile uyabilir (aşağıdaki şekil):


In [ ]:
# gmm_stretched_full.py
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
plot_gmm(gmm, X_stretched)



Bu, daha önce k-means ile karşılaşılan iki ana pratik sorunu GMM'nin nasıl giderdiğini netleştirir.

## Kovaryans Tipini Seçmek

Önceki uyumlara bakarsanız covariance_type seçeneğinin her birinde farklı ayarlandığını görürsünüz. Bu hiperparametre, her kümenin şeklindeki serbestlik derecelerini kontrol eder; verilen problem için dikkatle ayarlanmalıdır.

Varsayılan covariance_type="diag", her boyutta küme boyutunun bağımsız ayarlanabileceği anlamına gelir; elips eksenlerle hizalı kalır. Biraz daha basit ve hızlı model covariance_type="spherical", tüm boyutların eşit olmasını zorlar; sonuç k-means'e benzer ama tam eşdeğil değildir. Daha karmaşık ve pahalı model (boyut arttıkça özellikle) covariance_type="full", her kümeyi keyfi yönelimli elips olarak modellemeye izin verir.

Tek bir küme için bu üç seçeneğin görsel temsilini aşağıdaki şekilde görebiliriz:


In [ ]:
# make_moons_gmm.py
from sklearn.datasets import make_moons
Xmoon, ymoon = make_moons(200, noise=.05, random_state=0)
plt.scatter(Xmoon[:, 0], Xmoon[:, 1]);



## Yoğunluk Tahmini Olarak Gauss Karışım Modelleri

GMM sıklıkla kümeleme algoritması diye sınıflandırılsa da temelde bir yoğunluk tahmini algoritmasıdır. Yani GMM uyumu teknik olarak bir kümeleme modeli değil, verinin dağılımını tanımlayan üretken olasılıksal bir modeldir.

Örnek olarak Scikit-Learn'in make_moons fonksiyonundan üretilmiş veriyi düşünün (5.11 K-Means bölümünde tanıtılmıştı; aşağıdaki şekil):


In [ ]:
# gmm2_moons_fail.py
gmm2 = GaussianMixture(n_components=2, covariance_type='full', random_state=0)
plot_gmm(gmm2, Xmoon)



Bunu kümeleme modeli olarak iki bileşenli GMM ile uydurmaya çalışırsak sonuç pek kullanışlı olmaz (aşağıdaki şekil):


In [ ]:
# gmm16_density.py
gmm16 = GaussianMixture(n_components=16, covariance_type='full', random_state=0)
plot_gmm(gmm16, Xmoon, label=False)



Ancak çok daha fazla bileşen kullanıp küme etiketlerini yok sayarsak, girdi verisine çok daha yakın bir uyum buluruz (aşağıdaki şekil):


In [ ]:
# gmm16_sample.py
Xnew, ynew = gmm16.sample(400)
plt.scatter(Xnew[:, 0], Xnew[:, 1]);



Burada 16 Gauss bileşenli karışım, ayrık kümeler bulmak için değil, girdi verisinin genel dağılımını modellemek içindir. Bu, girdi verimize benzer yeni rastgele veri üretmek için tarif veren üretken bir dağılım modelidir. Örneğin orijinal veriye uydurulmuş 16 bileşenli GMM'den 400 yeni nokta çizebiliriz (aşağıdaki şekil):

GMM, keyfi çok boyutlu veri dağılımını modellemek için esnek bir araçtır.


In [ ]:
# gmm_aic_bic.py
n_components = np.arange(1, 21)
models = [GaussianMixture(n, covariance_type='full', random_state=0).fit(Xmoon)
          for n in n_components]

plt.plot(n_components, [m.bic(Xmoon) for m in models], label='BIC')
plt.plot(n_components, [m.aic(Xmoon) for m in models], label='AIC')
plt.legend(loc='best')
plt.xlabel('n_components');



### Kaç Bileşen?

GMM'nin üretken model olması, veri kümesi için optimal bileşen sayısını belirlemenin doğal bir yolunu verir. Üretken model doğası gereği veri kümesi için olasılık dağılımıdır; veriyi model altında değerlendirebiliriz, aşırı uyumu önlemek için çapraz doğrulama kullanırız.

Aşırı uyumu düzeltmek için AIC veya BIC gibi analitik ölçütlerle model olabilirliklerini ayarlayabiliriz. Scikit-Learn GaussianMixture tahmincisi her ikisini de hesaplayan yerleşik yöntemler içerir.

Ay verisi için GMM bileşen sayısına karşı AIC ve BIC'ye bakalım (aşağıdaki şekil):

> **Not**
>

Optimal küme sayısı AIC veya BIC'yi minimize eden değerdir. AIC, daha önce seçtiğimiz 16 bileşenin muhtemelen fazla olduğunu söyler: yaklaşık 8–12 bileşen daha iyi olurdu. Bu tür problemlerde BIC genelde daha sade model önerir.


In [ ]:
# load_digits_gmm.py
from sklearn.datasets import load_digits
digits = load_digits()
digits.data.shape



Hatırlamak için ilk 50 tanesini çizelim (aşağıdaki şekil):


In [ ]:
# plot_digits_fn.py
def plot_digits(data):
    fig, ax = plt.subplots(5, 10, figsize=(8, 4),
                           subplot_kw=dict(xticks=[], yticks=[]))
    fig.subplots_adjust(hspace=0.05, wspace=0.05)
    for i, axi in enumerate(ax.flat):
        im = axi.imshow(data[i].reshape(8, 8), cmap='binary')
        im.set_clim(0, 16)
plot_digits(digits.data)



64 boyutta yaklaşık 1.800 rakamımız var; bunun üzerine GMM kurabiliriz. GMM bu kadar yüksek boyutta yakınsamakta zorlanabilir; veri üzerinde tersinir bir boyut indirgeme algoritmasıyla başlayacağız. Burada doğrudan PCA kullanıp yansıtılan veride %99 varyansı koruyacağız:


In [ ]:
# pca_digits_gmm.py
from sklearn.decomposition import PCA
pca = PCA(0.99, whiten=True)
data = pca.fit_transform(digits.data)
data.shape



Sonuç 41 boyuttur — neredeyse 1/3 azalma, neredeyse bilgi kaybı yok. Bu yansıtılmış veriyle kaç GMM bileşeni kullanmamız gerektiğine AIC ile bakalım (aşağıdaki şekil):


In [ ]:
# gmm_aic_digits.py
n_components = np.arange(50, 210, 10)
models = [GaussianMixture(n, covariance_type='full', random_state=0)
          for n in n_components]
aics = [model.fit(data).aic(data) for model in models]
plt.plot(n_components, aics);



Yaklaşık 140 bileşen AIC'yi minimize ediyor gibi görünüyor; bu modeli kullanacağız. Veriye hızlıca uydurup yakınsadığını doğrulayalım:


In [ ]:
# gmm140_fit.py
gmm = GaussianMixture(140, covariance_type='full', random_state=0)
gmm.fit(data)
print(gmm.converged_)



Şimdi GMM'yi üretken model olarak kullanarak bu 41 boyutlu yansıtılmış uzayda 100 yeni nokta çekebiliriz:


In [ ]:
# gmm_sample_latent.py
data_new, label_new = gmm.sample(100)
data_new.shape



Son olarak PCA nesnesinin ters dönüşümüyle yeni rakamları oluşturabiliriz (aşağıdaki şekil):


In [ ]:
# gmm_new_digits.py
digits_new = pca.inverse_transform(data_new)
plot_digits(digits_new)



Sonuçların çoğu veri kümesindeki makul rakamlara benziyor! Burada yaptığımızı düşünün: el yazısı rakam örnekleminden yola çıkarak verinin dağılımını, karışım modeliyle girdi verisinde bireysel olarak görünmeyen ama genel özellikleri yakalayan yeni rakam örnekleri üretebilecek şekilde modelledik. Böyle bir üretken rakam modeli, sonraki bölümde göreceğimiz gibi Bayes üretken sınıflandırıcının bileşeni olarak çok faydalı olabilir.

> **Not**
>
